# Go2 / Go2W Real Robot `cmd_vel` Tutorial

**A line-by-line guide to understanding the hardware setup, network communication, ROS2 bridging, and performing basic `cmd_vel` operations from your laptop.**

---

## Table of Contents

1. [Hardware Overview: Go2 vs Go2W](#1)
2. [Physical Connection & Network Setup](#2)
3. [DDS Middleware Configuration (CycloneDDS)](#3)
4. [Sourcing Your Workspace](#4)
5. [The Go2W Driver & Service Calls](#5)
6. [The Full `cmd_vel` Pipeline (End-to-End)](#6)
7. [Hands-On: Sending Velocity Commands From Your Laptop](#7)
8. [Joystick Fallback & the Activity Mux](#8)
9. [Launching the Full Real-Robot Stack](#9)
10. [Troubleshooting & Tips](#10)

<a id='1'></a>
## 1. Hardware Overview: Go2 vs Go2W

### Unitree Go2 (Legged)
- **12 DOF** quadruped (3 joints per leg: hip, thigh, calf)
- Locomotion is purely legged — controlled via effort-based joint trajectories
- In simulation, uses CHAMP quadruped controller + Gazebo effort controllers

### Unitree Go2W (Wheeled)
- **16 DOF** — same 12 leg joints + **4 wheel joints** (one per foot)
- Can walk (legged mode) AND roll (wheel mode)
- The wheel joints are: `FL_foot_joint`, `FR_foot_joint`, `RL_foot_joint`, `RR_foot_joint`
- On the **real robot**, the Unitree driver handles both legged and wheeled motion internally when you send `/cmd_vel`
- In **Gazebo simulation**, we add a custom hybrid router (`go2w_hybrid_cmd_router.py`) that splits `cmd_vel` into legged vs wheel commands

### Key Joint Names

| Go2 (sim) | Go2W (sim & real) |
|---|---|
| `lf_hip_joint` | `FL_hip_joint` |
| `lf_upper_leg_joint` | `FL_thigh_joint` |
| `lf_lower_leg_joint` | `FL_calf_joint` |
| ... (x4 legs) | ... (x4 legs) + 4 wheel joints |

### Onboard Sensors
- **L1 Unitree LiDAR** — publishes `/utlidar/cloud` (PointCloud2) and `/utlidar/imu`
- **Joint encoders** — published as `joint_states` (JointState)
- **Onboard IMU** — published as `imu` (unitree_go/IMUState)
- **Wheel odometry** — published as `odom` (Odometry)

<a id='2'></a>
## 2. Physical Connection & Network Setup

### Wiring

```
+------------------+         Ethernet cable         +------------------+
|   Your Laptop    | <-----------------------------> |   Go2W Robot     |
|  192.168.123.100 |         (direct, no router)     |  192.168.123.18  |
|  enp3s0 / eth0   |                                 |  backward port   |
+------------------+                                 +------------------+
```

### Step-by-step Network Configuration

1. **Plug Ethernet cable** into the Go2W's **backward** Ethernet port (the one facing the tail)
2. **Set your laptop's IP** on the Ethernet interface:

```bash
# Find your Ethernet interface name (e.g. enp3s0, eth0, eno1)
ip link show

# Set static IP
sudo ip addr add 192.168.123.100/24 dev enp3s0
sudo ip link set enp3s0 up
```

3. **Verify connectivity:**

```bash
ping 192.168.123.18   # Should get replies from Go2W
```

### Important Notes
- The Go2W's backward Ethernet port is **always** `192.168.123.18` — this is hardcoded by Unitree, do NOT change it
- Your laptop **must** be `192.168.123.100` with netmask `255.255.255.0` (i.e. `/24`)
- Gateway: `192.168.123.1`
- No WiFi router needed — it's a direct point-to-point link
- If you also want WiFi for internet access, that's fine — just make sure DDS binds to the right interface (see next section)

<a id='3'></a>
## 3. DDS Middleware Configuration

ROS2 Humble uses DDS for communication. For the real robot, we use **CycloneDDS** and bind it to the Ethernet interface so ROS2 traffic goes over the direct cable link.

### CycloneDDS Setup

This is what `unitree_setup.sh` does under the hood:

```bash
# Switch ROS2 to CycloneDDS (instead of default FastDDS)
export RMW_IMPLEMENTATION=rmw_cyclonedds_cpp

# Tell CycloneDDS to ONLY use the Ethernet interface connected to Go2W
# Change 'enp3s0' to YOUR Ethernet interface name!
export CYCLONEDDS_URI='<CycloneDDS><Domain><General><Interfaces>
    <NetworkInterface name="enp3s0" priority="default" multicast="default" />
</Interfaces></General></Domain></CycloneDDS>'
```

### Why CycloneDDS?
- **FastDDS** (the ROS2 Humble default) uses shared memory (SHM) transport by default, which causes port conflicts when robot and laptop have separate memory spaces
- **CycloneDDS** with explicit interface binding ensures all DDS discovery and data goes over the Ethernet cable
- The repo also includes `fastdds_no_shm.xml` for Gazebo sim (disables SHM for FastDDS), but for real robot we prefer CycloneDDS

### Alternative: FastDDS without SHM (simulation only)

```bash
# Used by run_cfpa2_go2w_gazebo.sh for simulation
export FASTRTPS_DEFAULT_PROFILES_FILE="$(pwd)/fastdds_no_shm.xml"
```

### Verifying DDS

```bash
# After sourcing and exporting, check that you can see robot topics
ros2 topic list
# You should see topics like /cmd_vel, /odom, /joint_states, etc.
```

<a id='4'></a>
## 4. Sourcing Your Workspace

Before any ROS2 command, you need to source the right setup files. Here's the exact sequence:

```bash
# 1. Activate conda environment (needed for Python dependencies)
conda activate cmu_env
# or: micromamba activate cmu_env

# 2. Source ROS2 Humble
source /opt/ros/humble/setup.bash

# 3. Source your workspace overlay
source ~/COMP0225_LRC_stack/install/setup.bash

# 4. Configure DDS for real robot (CycloneDDS over Ethernet)
export RMW_IMPLEMENTATION=rmw_cyclonedds_cpp
export CYCLONEDDS_URI='<CycloneDDS><Domain><General><Interfaces>
    <NetworkInterface name="enp3s0" priority="default" multicast="default" />
</Interfaces></General></Domain></CycloneDDS>'
```

Or use the provided setup script:

```bash
source ~/COMP0225_LRC_stack/src/autonomy_stack_go2/unitree_setup.sh
```

**Every terminal** you open needs these exports. Missing them is the #1 cause of "I can't see the robot's topics".

<a id='5'></a>
## 5. The Go2W Driver & Service Calls

### What the Driver Does

The Go2W driver (`go2w_driver` C++ node from `unitree_go2w_ros2` package) is the bridge between ROS2 and the Unitree hardware SDK.

**It subscribes to:**

| Topic | Type | Purpose |
|-------|------|---------|
| `/cmd_vel` | `geometry_msgs/Twist` | **Velocity commands** — this is what moves the robot |
| `/utlidar/cloud` | `sensor_msgs/PointCloud2` | LiDAR point cloud from hardware SDK |
| `/utlidar/robot_pose` | `geometry_msgs/PoseStamped` | Pose from LiDAR-based odometry |
| `lowstate` | `unitree_go/LowState` | Raw joint state from motors |

**It publishes:**

| Topic | Type | Purpose |
|-------|------|---------|
| `pointcloud` | `sensor_msgs/PointCloud2` | Rebroadcasted point cloud |
| `joint_states` | `sensor_msgs/JointState` | Joint positions/velocities |
| `odom` | `nav_msgs/Odometry` | Wheel odometry |
| `imu` | IMU data | Onboard IMU readings |

### Critical Services

Before you can send `cmd_vel`, you **must** call two services:

#### 1. `/switch_joystick` — Disable the physical remote control

```bash
# flag=false means: "stop listening to the handheld remote, listen to ROS2 instead"
ros2 service call /switch_joystick go2_interfaces/srv/SwitchJoystick "{flag: false}"
```

If you skip this, the physical remote controller overrides all your ROS2 commands!

#### 2. `/mode` — Set the robot's posture/gait

```bash
# Make the robot stand up
ros2 service call /mode go2_interfaces/srv/Mode "{mode: 'stand_up'}"
```

Available modes:

| Mode | Description |
|------|-------------|
| `stand_up` | Stand from sitting/lying position |
| `sit` | Sit down |
| `damp` | Damping mode (limp, safe for handling) |
| `balance_stand` | Active balancing while standing |

### Automated Startup (what the launch file does)

The `go2w_startup_mode.py` node automates this at launch time:
1. Waits for `/switch_joystick` service to appear (up to 30s)
2. Calls `/switch_joystick` with `flag=false` (enables ROS2 control)
3. Waits for `/mode` service
4. Calls `/mode` with `stand_up`
5. Exits once both succeed

**File:** `src/go2_gazebo_sim/scripts/control/go2w_startup_mode.py`

<a id='6'></a>
## 6. The Full `cmd_vel` Pipeline (End-to-End)

Here's how a velocity command flows from the autonomy stack all the way to the robot's motors:

```
+----------------------------------------------------------------------+
|                        YOUR LAPTOP                                    |
|                                                                       |
|  +-------------------+                                                |
|  | CFPA2 Coordinator |  Picks frontier goals                          |
|  +---------+---------+                                                |
|            | /robot/way_point_coord (PointStamped)                    |
|            v                                                          |
|  +-------------------+                                                |
|  | Reactive Nav      |  Plans path, avoids obstacles                  |
|  | (reactive_nav.py) |  Produces velocity commands                    |
|  +---------+---------+                                                |
|            | /robot/cmd_vel_stamped (TwistStamped)                    |
|            v                                                          |
|  +-------------------+                                                |
|  | Twist Bridge      |  Strips timestamp header                       |
|  | (twist_bridge.py) |  TwistStamped -> Twist                        |
|  +---------+---------+                                                |
|            | /robot/cmd_vel_auto (Twist)                              |
|            v                                                          |
|  +-------------------+     +----------------+                         |
|  | Activity Mux      |<----| Joystick       | /robot/cmd_vel_manual   |
|  | (cmd_vel_activity  |     | (teleop_joy)  |                         |
|  |  _mux.py)         |     +----------------+                         |
|  +---------+---------+                                                |
|            | /cmd_vel (Twist) -- FINAL OUTPUT                         |
|            |                                                          |
+------------+----------------------------------------------------------+
             |  <-- Ethernet cable (192.168.123.x) via CycloneDDS -->
+------------+----------------------------------------------------------+
|            v                          GO2W ROBOT                      |
|  +-------------------+                                                |
|  | Go2W Driver       |  Converts Twist to unitree_api motor commands  |
|  | (go2w_driver)     |  via unitree_sdk2                              |
|  +---------+---------+                                                |
|            |                                                          |
|            v                                                          |
|  +-------------------+                                                |
|  | 12 leg motors     |  Walk / turn / balance                         |
|  | + 4 wheel motors  |  Roll forward/backward                         |
|  +-------------------+                                                |
+----------------------------------------------------------------------+
```

### Key Design Decisions

1. **Why the Twist Bridge?** The CMU autonomy stack (reactive_nav) outputs `TwistStamped` (with header timestamp), but the Go2W driver expects plain `Twist`. The bridge simply copies `.twist.linear` and `.twist.angular` and drops the header.

2. **Why the Activity Mux?** Safety. If a human grabs the joystick, manual commands immediately override autonomy. When the joystick goes idle for 0.35s, autonomy resumes. This prevents the robot from fighting the operator.

3. **Why `/cmd_vel` is a global topic** (no namespace): The Go2W driver subscribes to `/cmd_vel` without a namespace prefix. The mux's output topic is configured to `/cmd_vel` (global) so it reaches the driver directly.

4. **Real robot vs Simulation difference**: On the real robot, `/cmd_vel` goes directly to the Unitree driver. In Gazebo simulation, there's an extra `go2w_hybrid_cmd_router` that splits the command into legged (CHAMP) and wheel (ros2_control velocity) paths.

<a id='7'></a>
## 7. Hands-On: Sending Velocity Commands From Your Laptop

### Prerequisites Checklist

- [ ] Ethernet cable connected (laptop <-> Go2W backward port)
- [ ] Laptop IP set to `192.168.123.100/24`
- [ ] `ping 192.168.123.18` succeeds
- [ ] Go2W powered on and standing on a flat surface
- [ ] Workspace sourced + CycloneDDS configured (see Section 4)

### Method 1: Manual `cmd_vel` via command line

Open **three terminals**. Source the workspace in each (Section 4).

---

#### Terminal 1: Verify topics are visible

```bash
# Should list topics from the Go2W driver
ros2 topic list

# Expected output includes:
# /cmd_vel
# /odom
# /joint_states
# /pointcloud
# /utlidar/cloud
```

#### Terminal 2: Disable remote & stand up

```bash
# Step 1: Disable the physical remote controller
ros2 service call /switch_joystick go2_interfaces/srv/SwitchJoystick "{flag: false}"
# Expected: success=True

# Step 2: Command the robot to stand up
ros2 service call /mode go2_interfaces/srv/Mode "{mode: 'stand_up'}"
# Expected: success=True
# The robot should now physically stand up!
```

#### Terminal 3: Send velocity commands

```bash
# SAFETY: Start with very small velocities!

# Move forward slowly (0.1 m/s)
ros2 topic pub /cmd_vel geometry_msgs/msg/Twist \
  "{linear: {x: 0.1, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.0}}" \
  --rate 10

# Press Ctrl+C to stop publishing (robot will coast to stop)

# Turn left in place (0.3 rad/s)
ros2 topic pub /cmd_vel geometry_msgs/msg/Twist \
  "{linear: {x: 0.0, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.3}}" \
  --rate 10

# Move forward + turn right (arc)
ros2 topic pub /cmd_vel geometry_msgs/msg/Twist \
  "{linear: {x: 0.15, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: -0.2}}" \
  --rate 10

# EMERGENCY STOP: publish zero velocity
ros2 topic pub /cmd_vel geometry_msgs/msg/Twist \
  "{linear: {x: 0.0, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.0}}" \
  --once
```

### Understanding the Twist Message

```
geometry_msgs/msg/Twist:
  linear:
    x: float  -> forward/backward (m/s)     positive = forward
    y: float  -> left/right strafe (m/s)    positive = left
    z: float  -> (unused for ground robot)
  angular:
    x: float  -> (unused)
    y: float  -> (unused)
    z: float  -> yaw rotation (rad/s)       positive = counter-clockwise
```

### Safe Speed Limits (from reactive_nav_real_go2w.yaml)

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `max_linear_speed` | 0.28 m/s | Max forward/backward speed |
| `max_angular_speed` | 0.90 rad/s | Max turning speed |

**For manual testing, stay well below these -- use 0.1 m/s linear and 0.3 rad/s angular max.**

### Method 2: Python script for controlled motion

In [ ]:
#!/usr/bin/env python3
"""Send cmd_vel to a real Go2/Go2W robot from your laptop.

Prerequisites:
  - Ethernet cable connected to Go2W backward port
  - Laptop IP: 192.168.123.100/24
  - CycloneDDS configured (see tutorial Section 3)
  - /switch_joystick called with flag=false
  - /mode called with 'stand_up'
"""

import time
import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Twist


class SimpleCmdVelSender(Node):
    def __init__(self):
        super().__init__('simple_cmd_vel_sender')
        # Publish to /cmd_vel -- the Go2W driver subscribes here
        self.pub = self.create_publisher(Twist, '/cmd_vel', 10)
        self.get_logger().info('cmd_vel publisher ready. Sending commands...')

    def send(self, linear_x: float, angular_z: float, duration_sec: float):
        """Send a velocity command for a fixed duration, then stop."""
        msg = Twist()
        msg.linear.x = linear_x
        msg.angular.z = angular_z

        self.get_logger().info(
            f'Sending: linear.x={linear_x:.2f} m/s, angular.z={angular_z:.2f} rad/s '
            f'for {duration_sec:.1f}s'
        )

        rate = self.create_rate(10)  # 10 Hz
        start = time.monotonic()
        while time.monotonic() - start < duration_sec:
            self.pub.publish(msg)
            rate.sleep()

        # Stop
        self.pub.publish(Twist())  # all zeros
        self.get_logger().info('Stopped.')


def main():
    rclpy.init()
    node = SimpleCmdVelSender()

    try:
        # Move forward 0.1 m/s for 2 seconds
        node.send(linear_x=0.1, angular_z=0.0, duration_sec=2.0)
        time.sleep(1.0)  # pause between commands

        # Turn left 0.3 rad/s for 2 seconds
        node.send(linear_x=0.0, angular_z=0.3, duration_sec=2.0)
        time.sleep(1.0)

        # Move backward 0.1 m/s for 2 seconds
        node.send(linear_x=-0.1, angular_z=0.0, duration_sec=2.0)

    except KeyboardInterrupt:
        # Emergency stop on Ctrl+C
        node.pub.publish(Twist())
        node.get_logger().warn('Emergency stop!')
    finally:
        node.destroy_node()
        rclpy.shutdown()


if __name__ == '__main__':
    main()

### Method 3: `teleop_twist_keyboard` (interactive)

```bash
# Install if needed
sudo apt install ros-humble-teleop-twist-keyboard

# Run -- use WASD-like keys to drive
ros2 run teleop_twist_keyboard teleop_twist_keyboard --ros-args -r /cmd_vel:=/cmd_vel
```

Key bindings:
- `i` = forward, `,` = backward
- `j` = turn left, `l` = turn right
- `u`/`o` = forward+turn arcs
- `k` = stop
- `q`/`z` = increase/decrease speed

<a id='8'></a>
## 8. Joystick Fallback & the Activity Mux

### What is the Activity Mux?

The `cmd_vel_activity_mux.py` node acts as a **safety switch** between autonomous navigation and manual joystick control.

```
/robot/cmd_vel_auto    --+
  (from reactive_nav)    |
                         +---> [Activity Mux] ---> /cmd_vel (to driver)
                         |
/robot/cmd_vel_manual  --+
  (from joystick)
```

### Priority Logic

1. **Manual has priority**: If the joystick has been active in the last 0.35s (`manual_timeout_sec`), manual commands are forwarded
2. **Auto fallback**: If no joystick activity for 0.35s AND autonomy has sent a command in the last 0.60s (`auto_timeout_sec`), auto commands are forwarded
3. **Idle**: If neither source is active, zero velocity is published (robot stops)

### Activity Detection

A joystick command is considered "active" when:
- `hypot(linear.x, linear.y) > 0.02` (linear threshold), OR
- `|angular.z| > 0.05` (angular threshold)

This prevents joystick drift/noise from stealing control from autonomy.

### Status Topic

```bash
# Monitor which source is currently active
ros2 topic echo /robot/control_source
# Outputs: "manual", "auto", or "idle"
```

### Joystick Configuration

The joystick mapping (from `teleop_twist_joy_go2w.yaml`):

| Axis/Button | Joystick | Function |
|-------------|----------|---------|
| Axis 1 | Left stick Y | Forward/backward (scale: 0.35 m/s) |
| Axis 0 | Left stick X | Left/right strafe (scale: 0.30 m/s) |
| Axis 2 | Right stick X | Yaw rotation (scale: 0.80 rad/s) |
| Button 10 | L1/LB | Enable (must hold) |
| Button 8 | L2/LT | Turbo (hold for 0.60/1.20 speeds) |

**File:** `src/go2_gazebo_sim/config/nav/teleop_twist_joy_go2w.yaml`

<a id='9'></a>
## 9. Launching the Full Real-Robot Stack

### Quick Start (one command)

```bash
# Source everything first (Section 4), then:
source ~/COMP0225_LRC_stack/src/autonomy_stack_go2/unitree_setup.sh
source ~/COMP0225_LRC_stack/install/setup.bash

ros2 launch go2_gazebo_sim single_go2w_real_cfpa2.launch.py \
  robot_namespace:=robot \
  rviz:=true \
  enable_manual_fallback:=true
```

### What This Launches (in order)

| # | Node | What it does |
|---|------|--------------|
| 1 | `go2w_bringup` | Go2W driver + URDF description |
| 2 | `go2w_startup_mode` | Calls `/switch_joystick` + `/mode stand_up` |
| 3 | Point-LIO SLAM | LiDAR SLAM -> `/state_estimation` |
| 4 | `slam_odom_relay` | `/state_estimation` -> `/robot/odom/nav` |
| 5 | `pointcloud_to_laserscan` | 3D cloud -> 2D `/robot/scan_3d` |
| 6 | `simple_scan_mapper_cpp` | Builds occupancy grid `/robot/map` |
| 7 | `cfpa2_single_robot` | Frontier exploration coordinator |
| 8 | `reactive_nav` | Path planning + obstacle avoidance -> `/robot/cmd_vel_stamped` |
| 9 | `twist_bridge` | TwistStamped -> Twist (strips header) |
| 10 | `cmd_vel_activity_mux` | Manual/auto switching -> `/cmd_vel` |
| 11 | `joy_node` | (optional) Gamepad input |
| 12 | `teleop_twist_joy` | (optional) Joy -> `/robot/cmd_vel_manual` |

### Launch Arguments

```bash
# See all available arguments:
ros2 launch go2_gazebo_sim single_go2w_real_cfpa2.launch.py --show-args
```

| Argument | Default | Description |
|----------|---------|-------------|
| `robot_namespace` | `robot` | ROS2 namespace for all nodes |
| `rviz` | `false` | Open RViz visualization |
| `enable_manual_fallback` | `true` | Enable joystick override |
| `joy_dev` | `/dev/input/js0` | Gamepad device path |
| `startup_mode` | `stand_up` | Initial robot posture |
| `manual_timeout_sec` | `0.35` | Joystick activity timeout |
| `auto_timeout_sec` | `0.60` | Autonomy command timeout |

### Monitoring

```bash
# In a new (sourced) terminal:

# Watch odometry
ros2 topic echo /robot/odom/nav --field pose.pose.position

# Watch what's being sent to the robot
ros2 topic echo /cmd_vel

# Watch control source
ros2 topic echo /robot/control_source

# Check topic rates
ros2 topic hz /cmd_vel
ros2 topic hz /robot/scan_3d
```

<a id='10'></a>
## 10. Troubleshooting & Tips

### "I can't see any topics from the robot"

1. **Check network**: `ping 192.168.123.18` -- if no reply, check cable and IP config
2. **Check DDS**: Make sure `RMW_IMPLEMENTATION` and `CYCLONEDDS_URI` are exported
3. **Check interface name**: Run `ip link show` -- your Ethernet might be `eth0` not `enp3s0`. Update `CYCLONEDDS_URI` accordingly
4. **ROS_DOMAIN_ID**: Must match between laptop and robot (default is 0)

### "Robot ignores my cmd_vel"

1. **Did you call `/switch_joystick`?** The physical remote overrides ROS2 by default
2. **Did you call `/mode stand_up`?** The robot won't move in `damp` or `sit` mode
3. **Is the Activity Mux running?** If the full launch is active, your raw `/cmd_vel` might conflict with the mux's output. Either:
   - Kill the launch and publish to `/cmd_vel` directly, OR
   - Publish to `/robot/cmd_vel_manual` instead (the mux will pick it up)

### "Robot moves jerkily"

- Publish at **10+ Hz**. The driver expects continuous commands. Single messages cause start-stop behavior.
- Use `--rate 10` with `ros2 topic pub`, or use a timer in Python.

### "Robot drifts when I command zero"

- Small calibration offsets are normal. The Go2W driver has internal deadbands.
- If it's severe, check that `/cmd_vel` is actually receiving zeros: `ros2 topic echo /cmd_vel`

### Emergency Stop

```bash
# Method 1: Publish zero velocity
ros2 topic pub /cmd_vel geometry_msgs/msg/Twist \
  "{linear: {x: 0, y: 0, z: 0}, angular: {x: 0, y: 0, z: 0}}" --once

# Method 2: Put robot in damping mode (goes limp)
ros2 service call /mode go2_interfaces/srv/Mode "{mode: 'damp'}"

# Method 3: Use the physical remote controller (if /switch_joystick wasn't called)
# Press the "park" button on the Unitree remote
```

### Simulation vs Real Robot Summary

| Aspect | Simulation (Gazebo) | Real Robot |
|--------|--------------------|-----------|
| DDS | FastDDS (no SHM) | CycloneDDS over Ethernet |
| Driver | CHAMP + ros2_control | unitree_go2w_ros2 driver |
| Wheel control | Hybrid router splits cmd_vel | Driver handles internally |
| SLAM | Gazebo ground truth or Point-LIO | Point-LIO only |
| Stand up | `stand_up_slowly.py` (joint trajectory) | `/mode stand_up` service |
| Joystick disable | N/A (sim has no remote) | `/switch_joystick {flag: false}` |

---

## Quick Reference Card

```bash
# === SETUP (every terminal) ===
conda activate cmu_env
source /opt/ros/humble/setup.bash
source ~/COMP0225_LRC_stack/install/setup.bash
export RMW_IMPLEMENTATION=rmw_cyclonedds_cpp
export CYCLONEDDS_URI='<CycloneDDS><Domain><General><Interfaces>
    <NetworkInterface name="enp3s0" priority="default" multicast="default" />
</Interfaces></General></Domain></CycloneDDS>'

# === ENABLE ROS CONTROL ===
ros2 service call /switch_joystick go2_interfaces/srv/SwitchJoystick "{flag: false}"
ros2 service call /mode go2_interfaces/srv/Mode "{mode: 'stand_up'}"

# === DRIVE ===
ros2 topic pub /cmd_vel geometry_msgs/msg/Twist \
  "{linear: {x: 0.1, y: 0, z: 0}, angular: {x: 0, y: 0, z: 0}}" --rate 10

# === STOP ===
ros2 topic pub /cmd_vel geometry_msgs/msg/Twist \
  "{linear: {x: 0, y: 0, z: 0}, angular: {x: 0, y: 0, z: 0}}" --once

# === FULL AUTONOMY STACK ===
ros2 launch go2_gazebo_sim single_go2w_real_cfpa2.launch.py \
  robot_namespace:=robot rviz:=true enable_manual_fallback:=true
```